# Task-trained classifiers and multi-label heads on DENTEX tooth crops

This notebook produces Table 1, Table 2 and Fig. 2 of the manuscript.

Part 1 trains DenseNet-121, EfficientNetV2-S and ResNet-50 end to end on DENTEX tooth crops of the seed-42 radiograph-grouped split, and reports per-condition AUC, expected calibration error (ECE) and reliability diagrams for the best network.

Part 2 freezes the best Part 1 network as a feature extractor and compares three multi-label heads on the same features and split: a plain sigmoid head, independent per-condition classifiers and a gated MLP.

Each annotated tooth is one example. Annotations are grouped by radiograph, quadrant and tooth position, and the target is a four-condition multi-label vector (Impacted, Caries, Periapical Lesion, Deep Caries) read from `categories_3` of the DENTEX diagnosis file. All teeth of a radiograph are assigned to the same partition.

Set `DATA_DIR` in Section 2, then run all cells.

## 1. Environment and device

In [ ]:
import sys, subprocess
def pipq(*p): subprocess.run([sys.executable,"-m","pip","install","-q",*p], check=False)
# Packages are listed in requirements.txt. Uncomment to install them into the active kernel.
# pipq("torch","torchvision","scikit-learn","pillow","matplotlib","tqdm")

import torch, torchvision
print("torch:", torch.__version__, "| torchvision:", torchvision.__version__)

if torch.backends.mps.is_available():
    DEVICE = "mps"          # Apple silicon GPU
elif torch.cuda.is_available():
    DEVICE = "cuda"         # NVIDIA GPU
else:
    DEVICE = "cpu"
print("device:", DEVICE)
if DEVICE == "cpu":
    print("WARNING: no GPU backend found, so training will be slow.")


## 2. Data path

Set `DATA_DIR` to the DENTEX `quadrant-enumeration-disease` folder. The cell locates the image folder and the diagnosis JSON (the file whose annotations carry `category_id_3`).

In [ ]:
import os, glob, json

# DENTEX quadrant-enumeration-disease folder
DATA_DIR = "/path/to/DENTEX/training_data/quadrant-enumeration-disease"

assert os.path.isdir(DATA_DIR), f"Folder not found:\n{DATA_DIR}\nFix the path above."

# locate the xrays folder
cand = [DATA_DIR, os.path.join(DATA_DIR, "xrays")]
IMG_DIR = next((c for c in cand if os.path.isdir(c) and
                glob.glob(os.path.join(c, "*.png")) + glob.glob(os.path.join(c, "*.jpg"))), None)
if IMG_DIR is None:  # search one level down
    for root, _, files in os.walk(DATA_DIR):
        if any(f.lower().endswith((".png",".jpg")) for f in files):
            IMG_DIR = root; break
assert IMG_DIR, "Could not find an image folder with .png/.jpg files."

# locate the diagnosis JSON (must carry category_id_3)
ANN_PATH = None
for root, _, files in os.walk(DATA_DIR):
    for f in files:
        if f.endswith(".json"):
            try:
                j = json.load(open(os.path.join(root, f)))
            except Exception:
                continue
            if isinstance(j, dict) and "annotations" in j and \
               any("category_id_3" in a for a in j.get("annotations", [])):
                ANN_PATH = os.path.join(root, f); break
    if ANN_PATH: break
assert ANN_PATH, "Could not find the diagnosis JSON (needs category_id_3)."

print("Images :", IMG_DIR)
print("Labels :", ANN_PATH)
print("PNG count:", len(glob.glob(os.path.join(IMG_DIR, '*.png'))))

OUTPUT_DIR = os.path.join(DATA_DIR, "_results")
os.makedirs(os.path.join(OUTPUT_DIR, "checkpoints"), exist_ok=True)
print("Outputs ->", OUTPUT_DIR)


## 3. Tooth-level multi-label targets

Annotations are grouped by (radiograph, quadrant, tooth position). When a tooth carries two diagnoses, both target entries are set and the two boxes are merged.

In [ ]:
import numpy as np
from collections import defaultdict

coco = json.load(open(ANN_PATH))

# disease class map from the file (authoritative)
cats3 = sorted(coco["categories_3"], key=lambda c: c["id"])
DX_NAMES = [c["name"] for c in cats3]
NUM_DX = len(DX_NAMES)
id2idx = {c["id"]: i for i, c in enumerate(cats3)}
print("Classes:", {i: n for i, n in enumerate(DX_NAMES)})

img_file = {im["id"]: im["file_name"] for im in coco["images"]}

tooth = {}   # (image_id, q, pos) -> dict(image_id, file, bbox, y)
for a in coco["annotations"]:
    key = (a["image_id"], a.get("category_id_1", -1), a.get("category_id_2", -1))
    if key not in tooth:
        tooth[key] = {"image_id": a["image_id"],
                      "file": img_file[a["image_id"]],
                      "bbox": list(a["bbox"]),
                      "y": np.zeros(NUM_DX, np.float32)}
    tooth[key]["y"][id2idx[a["category_id_3"]]] = 1.0
    # merge bboxes (xywh)
    bx,by,bw,bh = tooth[key]["bbox"]; ax,ay,aw,ah = a["bbox"]
    x0,y0 = min(bx,ax), min(by,ay); x1,y1 = max(bx+bw,ax+aw), max(by+bh,ay+ah)
    tooth[key]["bbox"] = [x0,y0,x1-x0,y1-y0]

teeth = list(tooth.values())
Y = np.stack([t["y"] for t in teeth])
print(f"Teeth: {len(teeth)} | from {len(coco['images'])} images and {len(coco['annotations'])} annotations")
print("Positives per class:", dict(zip(DX_NAMES, Y.sum(0).astype(int).tolist())))
print("Multi-label teeth (>1 diagnosis):", int((Y.sum(1) > 1).sum()))


## 4. Radiograph-grouped split

The split is made by radiograph, so no radiograph contributes teeth to more than one of the training, validation and test partitions. It is deterministic for a given `SEED`; the reported results use `SEED = 42`.

In [ ]:
SEED = 42
VAL_FRAC, TEST_FRAC = 0.15, 0.15

def grouped_split(teeth, seed, val_frac, test_frac):
    img_ids = sorted({t["image_id"] for t in teeth})
    rng = np.random.RandomState(seed); rng.shuffle(img_ids)
    n = len(img_ids); n_te = int(n*test_frac); n_va = int(n*val_frac)
    test_imgs = set(img_ids[:n_te]); val_imgs = set(img_ids[n_te:n_te+n_va])
    tr, va, te = [], [], []
    for i, t in enumerate(teeth):
        if   t["image_id"] in test_imgs: te.append(i)
        elif t["image_id"] in val_imgs:  va.append(i)
        else:                            tr.append(i)
    return tr, va, te

train_idx, val_idx, test_idx = grouped_split(teeth, SEED, VAL_FRAC, TEST_FRAC)

# leakage assertion (fails loudly if an image ever crosses splits)
def _imgs(ix): return {teeth[i]["image_id"] for i in ix}
assert _imgs(train_idx).isdisjoint(_imgs(val_idx))
assert _imgs(train_idx).isdisjoint(_imgs(test_idx))
assert _imgs(val_idx).isdisjoint(_imgs(test_idx))
print(f"seed={SEED}  teeth: train={len(train_idx)} val={len(val_idx)} test={len(test_idx)}  (no image shared)")


## 5. Tooth-crop dataset

Each tooth is cropped from its padded bounding box on the full radiograph, converted to three-channel greyscale and resized to `CROP` pixels. Training crops receive light augmentation.

In [ ]:
from PIL import Image
import torchvision.transforms as T
from torch.utils.data import Dataset, DataLoader

CROP = 224          # tooth-crop size fed to the backbone
PAD  = 0.15         # bbox padding fraction
NORM = T.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225])

train_tf = T.Compose([T.Grayscale(3), T.Resize((CROP,CROP)),
                      T.RandomHorizontalFlip(0.5),
                      T.ColorJitter(brightness=0.1, contrast=0.1),
                      T.ToTensor(), NORM])
eval_tf  = T.Compose([T.Grayscale(3), T.Resize((CROP,CROP)), T.ToTensor(), NORM])

_img_cache = {}
def load_image(fname):
    if fname not in _img_cache:
        _img_cache[fname] = Image.open(os.path.join(IMG_DIR, os.path.basename(fname))).convert("L")
    return _img_cache[fname]

def crop_tooth(img, bbox, pad=PAD):
    W,H = img.size; x,y,w,h = bbox
    px,py = w*pad, h*pad
    x0=max(0,int(x-px)); y0=max(0,int(y-py)); x1=min(W,int(x+w+px)); y1=min(H,int(y+h+py))
    if x1<=x0 or y1<=y0: return img
    return img.crop((x0,y0,x1,y1))

class ToothDS(Dataset):
    def __init__(self, idxs, tf): self.idxs=list(idxs); self.tf=tf; self.c={}
    def __len__(self): return len(self.idxs)
    def __getitem__(self, k):
        i = self.idxs[k]; t = teeth[i]
        if i not in self.c:
            self.c[i] = crop_tooth(load_image(t["file"]), t["bbox"])
        return self.tf(self.c[i]), torch.from_numpy(t["y"])

BATCH = 32
def make_loaders():
    return (DataLoader(ToothDS(train_idx, train_tf), batch_size=BATCH, shuffle=True),
            DataLoader(ToothDS(val_idx,   eval_tf), batch_size=BATCH),
            DataLoader(ToothDS(test_idx,  eval_tf), batch_size=BATCH))
train_dl, val_dl, test_dl = make_loaders()
xb, yb = next(iter(train_dl))
print("batch:", tuple(xb.shape), tuple(yb.shape))


## 6. Metrics: per-condition AUC, ECE and reliability curves

In [ ]:
from sklearn.metrics import roc_auc_score, average_precision_score

@torch.no_grad()
def predict(model, dl):
    model.eval(); P=[]; Tt=[]
    for x,y in dl:
        p = torch.sigmoid(model(x.to(DEVICE))).cpu().numpy()
        P.append(p); Tt.append(y.numpy())
    return np.concatenate(P), np.concatenate(Tt)

def per_condition_auc(yt, yp):
    out={}
    for c in range(NUM_DX):
        out[DX_NAMES[c]] = roc_auc_score(yt[:,c], yp[:,c]) if len(np.unique(yt[:,c]))==2 else float("nan")
    return out

def macro_auc(yt, yp):
    v=[a for a in per_condition_auc(yt,yp).values() if not np.isnan(a)]
    return float(np.mean(v)) if v else float("nan")

def ece(yt, yp, n_bins=10):
    bins=np.linspace(0,1,n_bins+1); e=0.0
    for lo,hi in zip(bins[:-1],bins[1:]):
        m=(yp>lo)&(yp<=hi)
        if m.sum()==0: continue
        e += (m.sum()/len(yp))*abs(yp[m].mean()-yt[m].mean())
    return e


## 7. Training routine (used in Parts 1 and 2)

In [ ]:
import torch.nn as nn
from tqdm.auto import tqdm

# imbalance-aware positive weights from the TRAIN split
_pos = Y[train_idx].sum(0); _neg = len(train_idx) - _pos
POS_WEIGHT = torch.tensor(np.clip(_neg/np.maximum(_pos,1), 1.0, 20.0),
                          dtype=torch.float32, device=DEVICE)

def train_model(model, epochs=12, lr=1e-4, tag="model"):
    model = model.to(DEVICE)
    crit = nn.BCEWithLogitsLoss(pos_weight=POS_WEIGHT)
    opt  = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    best, best_state, hist = -1.0, None, []
    for ep in range(1, epochs+1):
        model.train(); run=0.0
        for x,y in tqdm(train_dl, desc=f"{tag} ep{ep}/{epochs}", leave=False):
            x,y = x.to(DEVICE), y.to(DEVICE)
            opt.zero_grad(); loss=crit(model(x), y); loss.backward(); opt.step()
            run += loss.item()*x.size(0)
        yp,yt = predict(model, val_dl); va = macro_auc(yt,yp)
        hist.append({"epoch":ep, "train_loss":run/len(train_idx), "val_macro_auc":va})
        print(f"  {tag} ep{ep:02d}  loss={run/len(train_idx):.4f}  val_macroAUC={va:.4f}")
        if va>best: best, best_state = va, {k:v.detach().cpu().clone() for k,v in model.state_dict().items()}
    if best_state: model.load_state_dict(best_state)
    return model, best, hist


---
# Part 1. Task-trained classifiers (Table 1, Fig. 2)

The three networks are trained on the same split and compared on per-condition AUC and calibration.

## 8. Networks: DenseNet-121, EfficientNetV2-S and ResNet-50

In [ ]:
from torchvision import models

def build_backbone(name, num_classes):
    if name == "densenet121":
        m = models.densenet121(weights=models.DenseNet121_Weights.IMAGENET1K_V1)
        m.classifier = nn.Linear(m.classifier.in_features, num_classes)
    elif name == "efficientnet_v2_s":
        m = models.efficientnet_v2_s(weights=models.EfficientNet_V2_S_Weights.IMAGENET1K_V1)
        m.classifier[1] = nn.Linear(m.classifier[1].in_features, num_classes)
    elif name == "resnet50":
        m = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
        m.fc = nn.Linear(m.fc.in_features, num_classes)
    else:
        raise ValueError(name)
    return m

ZOO = ["densenet121", "efficientnet_v2_s", "resnet50"]
EPOCHS_D1 = 12     # epochs per network


## 9. Training and test evaluation

In [ ]:
zoo_results = {}
for name in ZOO:
    print(f"\n=== {name} ===")
    model, best_val, hist = train_model(build_backbone(name, NUM_DX), epochs=EPOCHS_D1, tag=name)
    yp, yt = predict(model, test_dl)
    zoo_results[name] = {
        "best_val_macro_auc": round(best_val,4),
        "test_macro_auc": round(macro_auc(yt,yp),4),
        "test_per_condition_auc": {k:(None if np.isnan(v) else round(v,4)) for k,v in per_condition_auc(yt,yp).items()},
        "test_ece": {DX_NAMES[c]: round(ece(yt[:,c],yp[:,c]),4) for c in range(NUM_DX)},
        "history": hist,
    }
    torch.save(model.state_dict(), os.path.join(OUTPUT_DIR,"checkpoints",f"{name}_seed{SEED}.pt"))
    print(f"  -> test macroAUC = {zoo_results[name]['test_macro_auc']}")

print("\n=== Task-trained networks: test macro-AUC ===")
for n,r in zoo_results.items():
    print(f"{n:>20}: test macroAUC={r['test_macro_auc']}  per-cond={r['test_per_condition_auc']}")


## 10. Reliability diagrams of the best network (Fig. 2)

In [ ]:
import matplotlib.pyplot as plt

best_name = max(zoo_results, key=lambda n: zoo_results[n]["test_macro_auc"])
print("Best model:", best_name)
best_model = build_backbone(best_name, NUM_DX).to(DEVICE)
best_model.load_state_dict(torch.load(os.path.join(OUTPUT_DIR,"checkpoints",f"{best_name}_seed{SEED}.pt"),
                                      map_location=DEVICE))
yp, yt = predict(best_model, test_dl)

fig, axes = plt.subplots(1, NUM_DX, figsize=(4*NUM_DX, 3.6))
if NUM_DX==1: axes=[axes]
for c in range(NUM_DX):
    p=yp[:,c]; t=yt[:,c]; bins=np.linspace(0,1,11); xs=[]; ys=[]
    for lo,hi in zip(bins[:-1],bins[1:]):
        m=(p>lo)&(p<=hi)
        if m.sum()==0: continue
        xs.append(p[m].mean()); ys.append(t[m].mean())
    ax=axes[c]; ax.plot([0,1],[0,1],"--",color="gray",lw=1)
    ax.plot(xs,ys,"o-",color="#1a3a5c")
    ax.set_title(f"{DX_NAMES[c]}\nECE={ece(t,p):.3f}", fontsize=10)
    ax.set_xlabel("predicted"); ax.set_ylabel("empirical"); ax.set_xlim(0,1); ax.set_ylim(0,1)
plt.tight_layout(); plt.savefig(os.path.join(OUTPUT_DIR,f"d1_reliability_{best_name}_seed{SEED}.png"),dpi=130)
plt.show()


---
# Part 2. Multi-label head comparison (Table 2)

The best Part 1 network is frozen and used as a feature extractor. Three heads are trained on identical features and split: a plain sigmoid head, independent per-condition classifiers and a gated MLP.

## 11. Feature extraction with the frozen network

In [ ]:
import torch.nn as nn

def make_feature_extractor(name):
    m = build_backbone(name, NUM_DX)
    ck = os.path.join(OUTPUT_DIR,"checkpoints",f"{name}_seed{SEED}.pt")
    if os.path.exists(ck): m.load_state_dict(torch.load(ck, map_location="cpu"))
    # strip the classifier -> expose pooled features
    if name=="densenet121":
        feat = nn.Sequential(m.features, nn.ReLU(inplace=True),
                             nn.AdaptiveAvgPool2d(1), nn.Flatten())
        fdim = m.classifier.in_features
    elif name=="efficientnet_v2_s":
        feat = nn.Sequential(m.features, m.avgpool, nn.Flatten()); fdim = m.classifier[1].in_features
    else:  # resnet50
        feat = nn.Sequential(*list(m.children())[:-1], nn.Flatten()); fdim = m.fc.in_features
    return feat.to(DEVICE).eval(), fdim

feat_ext, FDIM = make_feature_extractor(best_name)
print("feature dim:", FDIM)

@torch.no_grad()
def extract(dl):
    F=[]; Y_=[]
    for x,y in tqdm(dl, desc="extract", leave=False):
        F.append(feat_ext(x.to(DEVICE)).cpu()); Y_.append(y)
    return torch.cat(F), torch.cat(Y_)

Ftr, Ytr = extract(train_dl)
Fva, Yva = extract(val_dl)
Fte, Yte = extract(test_dl)
print("features:", tuple(Ftr.shape), tuple(Fte.shape))


## 12. The three heads

In [ ]:
class PlainHead(nn.Module):
    def __init__(self, f, k): super().__init__(); self.l=nn.Linear(f,k)
    def forward(self,x): return self.l(x)

class IndepHead(nn.Module):
    def __init__(self, f, k):
        super().__init__()
        self.heads=nn.ModuleList([nn.Sequential(nn.Linear(f,64),nn.ReLU(),nn.Linear(64,1)) for _ in range(k)])
    def forward(self,x): return torch.cat([h(x) for h in self.heads],1)

# Gated MLP: a shared trunk, then for each condition a gate multiplied by an expert output.
class GatedMLPHead(nn.Module):
    def __init__(self, f, k, hidden=128):
        super().__init__()
        self.shared=nn.Sequential(nn.Linear(f,hidden),nn.ReLU(),nn.Dropout(0.2))
        self.gate  =nn.ModuleList([nn.Linear(hidden,1) for _ in range(k)])
        self.expert=nn.ModuleList([nn.Linear(hidden,1) for _ in range(k)])
    def forward(self,x):
        h=self.shared(x); outs=[]
        for g,e in zip(self.gate,self.expert):
            outs.append(torch.sigmoid(g(h))*e(h))
        return torch.cat(outs,1)
    def gate_values(self,x):
        with torch.no_grad():
            h=self.shared(x)
            return torch.cat([torch.sigmoid(g(h)) for g in self.gate],1)


## 13. Head training on the frozen features

In [ ]:
from torch.utils.data import TensorDataset, DataLoader

def loaders_from_feats(bs=64):
    return (DataLoader(TensorDataset(Ftr,Ytr), batch_size=bs, shuffle=True),
            DataLoader(TensorDataset(Fva,Yva), batch_size=bs),
            DataLoader(TensorDataset(Fte,Yte), batch_size=bs))
ftr_dl, fva_dl, fte_dl = loaders_from_feats()

def train_head(head, epochs=40, lr=1e-3, tag="head"):
    head=head.to(DEVICE)
    crit=nn.BCEWithLogitsLoss(pos_weight=POS_WEIGHT)
    opt=torch.optim.AdamW(head.parameters(), lr=lr, weight_decay=1e-4)
    best,best_state=-1.0,None
    for ep in range(epochs):
        head.train()
        for f,y in ftr_dl:
            f,y=f.to(DEVICE),y.to(DEVICE)
            opt.zero_grad(); loss=crit(head(f),y); loss.backward(); opt.step()
        head.eval()
        with torch.no_grad():
            yp=torch.sigmoid(head(Fva.to(DEVICE))).cpu().numpy()
        va=macro_auc(Yva.numpy(),yp)
        if va>best: best,best_state=va,{k:v.detach().cpu().clone() for k,v in head.state_dict().items()}
    if best_state: head.load_state_dict(best_state)
    return head,best

ablation={}
heads={"plain":PlainHead(FDIM,NUM_DX),"independent":IndepHead(FDIM,NUM_DX),"gated_mlp":GatedMLPHead(FDIM,NUM_DX)}
trained={}
for name,h in heads.items():
    hh,bv=train_head(h, tag=name)
    trained[name]=hh
    with torch.no_grad():
        yp=torch.sigmoid(hh(Fte.to(DEVICE))).cpu().numpy()
    yt=Yte.numpy()
    ablation[name]={"val_macro_auc":round(bv,4),"test_macro_auc":round(macro_auc(yt,yp),4),
                    "test_per_condition_auc":{k:(None if np.isnan(v) else round(v,4)) for k,v in per_condition_auc(yt,yp).items()}}
    print(f"{name:>12}: val={ablation[name]['val_macro_auc']}  test={ablation[name]['test_macro_auc']}")

print("\n=== Multi-label heads: test macro-AUC ===")
for n,r in ablation.items(): print(f"  {n:>12}: {r['test_macro_auc']}")


## 14. Save results

In [ ]:
import json, datetime
summary={
    "timestamp": datetime.datetime.now().isoformat(timespec="seconds"),
    "device": DEVICE, "seed": SEED, "crop": CROP,
    "n_teeth": len(teeth), "class_names": DX_NAMES,
    "positives_per_class": dict(zip(DX_NAMES, Y.sum(0).astype(int).tolist())),
    "split": {"train":len(train_idx),"val":len(val_idx),"test":len(test_idx)},
    "direction1_zoo": zoo_results,
    "direction1_best_model": best_name,
    "direction2_ablation": ablation,
    "notes": {
        "unit":"tooth (grouped by image,quadrant,position)",
        "targets":"multi-label over 4 DENTEX diagnoses",
        "split":"grouped by radiograph",
        "caveats":"small data (705 imgs); periapical lesion rarest/noisiest; gated MLP gains modest with only 4 classes",
    },
}
p=os.path.join(OUTPUT_DIR,f"summary_seed{SEED}.json")
json.dump(summary, open(p,"w"), indent=2)
print("Saved ->", p)
print(json.dumps({"d1_best":best_name,
                  "d1_test_macroAUC":zoo_results[best_name]["test_macro_auc"],
                  "d2_ablation":{k:v["test_macro_auc"] for k,v in ablation.items()}}, indent=2))


---
# Part 3. Bounded update on frozen EfficientNetV2-S features (Fig. 6)

The best Part 1 network (EfficientNetV2-S) is frozen, and the method is fitted to its 1,280-dimensional penultimate features from Section 11: a new logistic probe per condition, a temperature fitted on the validation split, and the clipped residual head with δ = 1.5. The reported run gave T = 3.4, macro-AUC 0.909 for the probe, and macro-ECE 0.056 for the probe and 0.062 after the residual update. Run Sections 1 to 11 first.

## 15. Probe, temperature and residual head on the frozen features

In [ ]:
# Reference probe, temperature and clipped residual head on frozen EfficientNetV2-S penultimate features
import numpy as np, torch, torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.linear_model import LogisticRegression
DELTA = 1.5

Xtr, Xva, Xte = Ftr.numpy(), Fva.numpy(), Fte.numpy()
ytr, yva, yte = Ytr.numpy(), Yva.numpy(), Yte.numpy()

def fit_baseline(X, Yb):
    return [LogisticRegression(max_iter=1000, C=1.0).fit(X, Yb[:,c]) if len(np.unique(Yb[:,c]))>1 else None
            for c in range(NUM_DX)]
def blogits(probes, X):
    out=np.zeros((len(X),NUM_DX),np.float32)
    for c,p in enumerate(probes): out[:,c]= p.decision_function(X) if p is not None else 0.0
    return out
def ece(y,p,nb=10):
    b=np.linspace(0,1,nb+1); e=0.0
    for lo,hi in zip(b[:-1],b[1:]):
        m=(p>lo)&(p<=hi)
        if m.sum(): e+=(m.sum()/len(p))*abs(p[m].mean()-y[m].mean())
    return e

probes = fit_baseline(Xtr, ytr)
# temperature on val
Tp=nn.Parameter(torch.ones(1)); z=torch.tensor(blogits(probes,Xva)); yt_=torch.tensor(yva)
opt=torch.optim.LBFGS([Tp],lr=0.1,max_iter=60); bce=nn.BCEWithLogitsLoss()
opt.step(lambda:(opt.zero_grad(), bce(z/Tp.clamp(min=0.05),yt_).backward(), bce(z/Tp.clamp(min=0.05),yt_))[-1])
T=float(Tp.detach().clamp(min=0.05))

class EnvelopeHead(nn.Module):
    def __init__(s,d,k,delta): super().__init__(); s.delta=delta; s.net=nn.Sequential(nn.Linear(d,256),nn.ReLU(),nn.Dropout(0.2),nn.Linear(256,k))
    def forward(s,x,base): return base + torch.clamp(s.net(x),-s.delta,s.delta)

head=EnvelopeHead(Xtr.shape[1],NUM_DX,DELTA).to(DEVICE)
btr=torch.tensor(blogits(probes,Xtr)/T); bva=torch.tensor(blogits(probes,Xva)/T)
dl=DataLoader(TensorDataset(torch.tensor(Xtr),btr,torch.tensor(ytr)),batch_size=64,shuffle=True)
op=torch.optim.AdamW(head.parameters(),lr=1e-3,weight_decay=1e-4); crit=nn.BCEWithLogitsLoss()
best=1e9; bs=None
for ep in range(60):
    head.train()
    for x,b,y in dl:
        x,b,y=x.to(DEVICE),b.to(DEVICE),y.to(DEVICE); op.zero_grad(); crit(head(x,b),y).backward(); op.step()
    head.eval()
    with torch.no_grad(): vl=crit(head(torch.tensor(Xva).to(DEVICE),bva.to(DEVICE)),torch.tensor(yva).to(DEVICE)).item()
    if vl<best: best=vl; bs={k:v.detach().cpu().clone() for k,v in head.state_dict().items()}
head.load_state_dict(bs); head.eval()
print(f"EfficientNet envelope fitted | T={T:.3f}")

## 16. Test-split AUC, ECE and maximum logit deviation

In [ ]:
from sklearn.metrics import roc_auc_score
def macro_auc(Y,P): 
    v=[roc_auc_score(Y[:,c],P[:,c]) for c in range(NUM_DX) if len(np.unique(Y[:,c]))==2]; return float(np.mean(v))
bte=blogits(probes,Xte)/T
p_base=1/(1+np.exp(-bte))
with torch.no_grad():
    final=head(torch.tensor(Xte).to(DEVICE),torch.tensor(bte).to(DEVICE)).cpu().numpy()
p_env=1/(1+np.exp(-final))
max_dev=float(np.max(np.abs(final-bte)))
print(f"EfficientNet  baseAUC {macro_auc(yte,p_base):.4f} -> envAUC {macro_auc(yte,p_env):.4f}")
print(f"              baseECE {np.mean([ece(yte[:,c],p_base[:,c]) for c in range(NUM_DX)]):.4f} -> "
      f"envECE {np.mean([ece(yte[:,c],p_env[:,c]) for c in range(NUM_DX)]):.4f}")
print(f"              guarantee max|dev|={max_dev:.4f} (<= {DELTA}): {max_dev<=DELTA+1e-4}")

## 17. Case figures on train_564 (Fig. 6)

The first cell draws the end-to-end EfficientNetV2-S outputs; the second draws the same teeth after the bounded update, with teeth beyond the 95th-percentile validation distance marked as refused. Both images are saved to `OUTPUT_DIR`.

In [ ]:
# EfficientNetV2-S outputs on train_564 without the bounded update or the gate (Fig. 6, first panel).
import numpy as np, torch, os
from PIL import Image, ImageDraw, ImageFont

TARGET = "train_564"
rows = [(i,t) for i,t in enumerate(teeth) if TARGET in os.path.basename(t["file"])]
print(f"{TARGET}: {len(rows)} teeth")
assert len(rows) > 0

best_model.eval()
base_gray = load_image(teeth[rows[0][0]]["file"])
img = base_gray.convert("RGB")
draw = ImageDraw.Draw(img)
try: font = ImageFont.truetype("/System/Library/Fonts/Helvetica.ttc", 26)
except: font = ImageFont.load_default()

COLORS = {0:(170,130,255),1:(240,200,40),2:(80,160,255),3:(255,70,70)}
THRESH = 0.5

def text_size(s):
    b = draw.textbbox((0,0), s, font=font)
    return b[2]-b[0], b[3]-b[1]

# first pass: collect findings
placed = []   # list of (x,y,w,h, label, color)
with torch.no_grad():
    for i,t in rows:
        crop = crop_tooth(base_gray, t["bbox"])
        x = eval_tf(crop).unsqueeze(0).to(DEVICE)
        p = torch.sigmoid(best_model(x))[0].cpu().numpy()
        x0,y0,w,h = t["bbox"]; x1,y1 = x0+w, y0+h
        top = int(np.argmax(p))
        if p[top] >= THRESH:
            placed.append([x0,y0,x1,y1, f"{DX_NAMES[top]} {p[top]*100:.0f}%", COLORS.get(top,(255,255,255))])
        else:
            draw.rectangle([x0,y0,x1,y1], outline=(90,200,90), width=2)

# second pass: draw boxes + labels, staggering labels that would overlap
label_rects = []   # occupied label regions (lx0,ly0,lx1,ly1)
def overlaps(r):
    for q in label_rects:
        if not (r[2] < q[0] or r[0] > q[2] or r[3] < q[1] or r[1] > q[3]):
            return True
    return False

# sort top-to-bottom, left-to-right so stagger is stable
placed.sort(key=lambda r:(r[1], r[0]))
for x0,y0,x1,y1,label,c in placed:
    draw.rectangle([x0,y0,x1,y1], outline=c, width=3)
    tw,th = text_size(label)
    # candidate label positions: above box, then nudge upward in steps until free
    lx = x0
    ly = y0 - th - 8
    tries = 0
    while (ly < 2 or overlaps((lx, ly, lx+tw+8, ly+th+6))) and tries < 12:
        ly -= (th + 8)
        if ly < 2:                       # ran off top -> place below the box instead
            ly = y1 + 6 + (tries* (th+8))
        tries += 1
    lx = max(2, min(lx, img.width - tw - 8))
    # filled background for legibility
    draw.rectangle([lx, ly, lx+tw+8, ly+th+6], fill=(0,0,0))
    draw.text((lx+4, ly+2), label, fill=c, font=font)
    label_rects.append((lx, ly, lx+tw+8, ly+th+6))

out = os.path.join(OUTPUT_DIR, "train_564_efficientnet_RAW.png")
img.save(out)
print(f"drew {len(placed)} findings -> {out}")

In [ ]:
# The same radiograph with the reference probe, clipped residual and Mahalanobis gate (Fig. 6, second panel).
import numpy as np, torch, os
from PIL import Image, ImageDraw, ImageFont

# Mahalanobis gate on the EfficientNetV2-S features
mu = Xtr.mean(0); cov = np.cov(Xtr.T) + np.eye(Xtr.shape[1])*1e-2; ci = np.linalg.inv(cov)
def maha(X): return np.einsum("ni,ij,nj->n", X-mu, ci, X-mu)
ood_thr = float(np.quantile(maha(Xva), 0.95))   # 95th percentile of the validation distances
print(f"EfficientNet OOD threshold (p95 val): {ood_thr:.1f}")

TARGET="train_564"
rows=[(i,t) for i,t in enumerate(teeth) if TARGET in os.path.basename(t["file"])]
print(f"{TARGET}: {len(rows)} teeth")

best_gray = load_image(teeth[rows[0][0]]["file"])
img = best_gray.convert("RGB"); draw=ImageDraw.Draw(img)
try: font=ImageFont.truetype("/System/Library/Fonts/Helvetica.ttc",26)
except: font=ImageFont.load_default()
COLORS={0:(170,130,255),1:(240,200,40),2:(80,160,255),3:(255,70,70)}
def tsize(s): b=draw.textbbox((0,0),s,font=font); return b[2]-b[0],b[3]-b[1]

# feature extractor (same as cell 24)
feat_ext_local, _ = (feat_ext, FDIM) if 'feat_ext' in globals() else make_feature_extractor(best_name)

placed=[]
with torch.no_grad():
    for i,t in rows:
        crop=crop_tooth(best_gray,t["bbox"])
        xf=eval_tf(crop).unsqueeze(0).to(DEVICE)
        emb=feat_ext_local(xf).cpu().numpy()                  # 1280-d feature
        d=float(maha(emb))                                    # OOD distance
        x0,y0,w,h=t["bbox"]; x1,y1=x0+w,y0+h
        if d>ood_thr:                                         # refused by the gate
            placed.append([x0,y0,x1,y1,"Deferred (OOD)",(220,220,220)])
            continue
        # probability after the bounded update: reference logit / T plus clipped residual
        bl = blogits(probes, emb)/T
        with torch.no_grad():
            final = head(torch.tensor(emb).to(DEVICE), torch.tensor(bl).to(DEVICE)).cpu().numpy()[0]
        p = 1/(1+np.exp(-final))
        top=int(np.argmax(p))
        if p[top]>=0.5:
            placed.append([x0,y0,x1,y1,f"{DX_NAMES[top]} {p[top]*100:.0f}%",COLORS.get(top,(255,255,255))])
        else:
            draw.rectangle([x0,y0,x1,y1],outline=(90,200,90),width=2)

# labels drawn as in the previous cell
label_rects=[]
def ov(r):
    return any(not (r[2]<q[0] or r[0]>q[2] or r[3]<q[1] or r[1]>q[3]) for q in label_rects)
placed.sort(key=lambda r:(r[1],r[0]))
for x0,y0,x1,y1,label,c in placed:
    draw.rectangle([x0,y0,x1,y1],outline=c,width=3)
    tw,th=tsize(label); lx=x0; ly=y0-th-8; tries=0
    while (ly<2 or ov((lx,ly,lx+tw+8,ly+th+6))) and tries<12:
        ly-=(th+8)
        if ly<2: ly=y1+6+tries*(th+8)
        tries+=1
    lx=max(2,min(lx,img.width-tw-8))
    draw.rectangle([lx,ly,lx+tw+8,ly+th+6],fill=(0,0,0))
    draw.text((lx+4,ly+2),label,fill=c,font=font)
    label_rects.append((lx,ly,lx+tw+8,ly+th+6))

out=os.path.join(OUTPUT_DIR,"train_564_efficientnet_ENVELOPED.png")
img.save(out); print(f"drew {len(placed)} accepted/deferred -> {out}")